# AutoGen 실습 : 미래 서울의 기억 배달부

In [1]:
%pip install -q -U openai autogen-agentchat "autogen-ext[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 11.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

OPENROUTER_API_KEY = userdata.get("OpenRouterapi")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print("OpenRouter API 키 준비 완료")

OpenRouter API 키 준비 완료


## 아이디어와 캐릭터

In [3]:
idea = '''
가까운 미래의 서울. 사람의 기억을 디지털 패키지처럼 안전하게 보관하고 전달하는 서비스가 존재한다.
기억 배달부 윤하는 의뢰받은 기억 조각을 밤의 서울 곳곳으로 전달하던 중, 자신이 잃어버린 과거와 연결된 수상한 기억 패키지를 발견한다.
윤하는 추적을 피하면서도 한 소녀에게 꼭 전해야 하는 기억의 정체를 밝혀야 한다.
'''

character_bank = '''
윤하: 미래 서울에서 활동하는 기억 배달부. 차분하고 신중하지만 자신의 과거 일부가 비어 있다는 사실을 늘 의식하고 있다.

도윤: 불법 기억 복원 기술자. 냉소적이고 말투는 거칠지만, 누구보다 빠르게 손상된 기억 데이터를 복구할 수 있다.

서진: 대기업 기억보관소의 보안 책임자. 원칙주의자처럼 보이지만, 사건을 쫓는 이유에 개인적인 사연이 숨어 있다.

하나: 어린 소녀. 어머니와 관련된 마지막 기억을 되찾고 싶어 하며, 이번 사건의 감정적 중심이 되는 인물이다.
'''

## 에이전트 시스템 프롬프트

In [4]:
system_message_scenario_writer = '''
당신은 영화 시나리오 작가입니다.
반드시 텍스트만 출력하세요. 이미지 마크다운, 코드블록, 표는 사용하지 마세요.

역할:
- 주어진 아이디어와 캐릭터 설정을 바탕으로 매력적인 전체 이야기의 뼈대를 만듭니다.
- 이야기를 3개에서 5개의 서브스크립트로 나눕니다.

출력 형식:
서브스크립트 1
Plot: ...
Involving Characters: ...

서브스크립트 2
Plot: ...
Involving Characters: ...

지침:
- 배경은 미래 서울의 밤 분위기를 충분히 살리세요.
- 감정선과 미스터리를 함께 가져가세요.
- 각 서브스크립트는 다음 에이전트가 장면으로 확장하기 쉽도록 명확해야 합니다.
'''

description_scenario_writer = "아이디어와 캐릭터를 바탕으로 영화 전체 시나리오 골격을 만드는 작가"

system_message_scene_planner = '''
당신은 영화 장면 기획자입니다.
반드시 텍스트만 출력하세요. 이미지 마크다운, 코드블록, 표는 사용하지 마세요.

역할:
- 이전 에이전트가 만든 시나리오를 읽고, 각 서브스크립트를 실제 촬영 가능한 장면 흐름으로 더 구체화합니다.
- 전체 이야기가 끊기지 않도록 3개에서 5개의 핵심 장면 단위로 정리합니다.

출력 형식:
장면 1
Plot: ...
Involving Characters: ...

장면 2
Plot: ...
Involving Characters: ...

지침:
- 사건 전개가 점점 긴장감을 높이도록 구성하세요.
- 장소 변화, 추적, 감정 폭발 지점을 분명히 드러내세요.
- 다음 에이전트가 바로 콘티를 만들 수 있도록 구체적으로 쓰세요.
'''

description_scene_planner = "시나리오를 장면 단위로 세분화하는 장면 기획자"

system_message_storyboard_artist = '''
당신은 영화 스토리보드 아티스트입니다.
반드시 텍스트만 출력하세요. 이미지 마크다운, 코드블록, 표는 사용하지 마세요.
최종 출력은 3900자 이하로 유지하세요.

역할:
- 이전 에이전트가 정리한 장면들을 읽고, 바로 이미지 생성에 활용할 수 있는 수준의 스토리보드로 완성합니다.
- 장면마다 분위기, 비주얼, 핵심 소품, 촬영 메모를 포함하세요.

출력 형식:
장면 1
Plot: ...
Scene Description: ...
Emotional Tone: ...
Visual Style: ...
Key Props: ...
Involving Characters: ...
Cinematography Notes: ...

장면 2
Plot: ...
Scene Description: ...
Emotional Tone: ...
Visual Style: ...
Key Props: ...
Involving Characters: ...
Cinematography Notes: ...

지침:
- 미래 서울의 네온, 골목, 고층 빌딩, 디지털 기억 장치 같은 시각 요소를 적극 반영하세요.
- 각 장면이 서로 이어지면서도 대표 이미지로 뽑히기 좋게 묘사하세요.
- 모든 장면을 통합한 뒤 맨 마지막 줄에 TERMINATE를 단독으로 출력하세요.
'''

description_storyboard_artist = "최종 촬영 콘티와 이미지 생성용 장면 설명을 만드는 스토리보드 아티스트"

## OpenRouter Nemotron 모델 클라이언트 정의

In [5]:
model_client = OpenAIChatCompletionClient(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "autogen_prac03",
    },
    model_info={
        "vision": False,
        "function_calling": False,
        "json_output": False,
        "structured_output": False,
        "family": "unknown",
    },
)

print("OpenRouter Nemotron 무료 모델 클라이언트 준비 완료")

OpenRouter Nemotron 무료 모델 클라이언트 준비 완료


## 에이전트 정의

In [6]:
scenario_writer_agent = AssistantAgent(
    name="scenario_writer_agent",
    model_client=model_client,
    description=description_scenario_writer,
    system_message=system_message_scenario_writer,
)

scene_planner_agent = AssistantAgent(
    name="scene_planner_agent",
    model_client=model_client,
    description=description_scene_planner,
    system_message=system_message_scene_planner,
)

storyboard_artist_agent = AssistantAgent(
    name="storyboard_artist_agent",
    model_client=model_client,
    description=description_storyboard_artist,
    system_message=system_message_storyboard_artist,
)

termination = TextMentionTermination("TERMINATE")

group_chat = RoundRobinGroupChat(
    [scenario_writer_agent, scene_planner_agent, storyboard_artist_agent],
    termination_condition=termination,
)

## 실행

In [7]:
movie_task = f'''
다음 아이디어와 캐릭터 설정을 바탕으로, 미래 서울을 배경으로 한 감성 SF 미스터리 영화 프로젝트를 완성해주세요.

[아이디어]
{idea}

[캐릭터 설정]
{character_bank}

최종 목표:
- 전체 이야기의 흐름이 자연스럽게 이어질 것
- 장면마다 감정과 비주얼이 선명할 것
- 마지막 스토리보드는 대표 이미지 생성에 바로 쓸 수 있을 정도로 구체적일 것
'''

result = await Console(group_chat.run_stream(task=movie_task))

---------- TextMessage (user) ----------

다음 아이디어와 캐릭터 설정을 바탕으로, 미래 서울을 배경으로 한 감성 SF 미스터리 영화 프로젝트를 완성해주세요.

[아이디어]

가까운 미래의 서울. 사람의 기억을 디지털 패키지처럼 안전하게 보관하고 전달하는 서비스가 존재한다.
기억 배달부 윤하는 의뢰받은 기억 조각을 밤의 서울 곳곳으로 전달하던 중, 자신이 잃어버린 과거와 연결된 수상한 기억 패키지를 발견한다.
윤하는 추적을 피하면서도 한 소녀에게 꼭 전해야 하는 기억의 정체를 밝혀야 한다.


[캐릭터 설정]

윤하: 미래 서울에서 활동하는 기억 배달부. 차분하고 신중하지만 자신의 과거 일부가 비어 있다는 사실을 늘 의식하고 있다.

도윤: 불법 기억 복원 기술자. 냉소적이고 말투는 거칠지만, 누구보다 빠르게 손상된 기억 데이터를 복구할 수 있다.

서진: 대기업 기억보관소의 보안 책임자. 원칙주의자처럼 보이지만, 사건을 쫓는 이유에 개인적인 사연이 숨어 있다.

하나: 어린 소녀. 어머니와 관련된 마지막 기억을 되찾고 싶어 하며, 이번 사건의 감정적 중심이 되는 인물이다.


최종 목표:
- 전체 이야기의 흐름이 자연스럽게 이어질 것
- 장면마다 감정과 비주얼이 선명할 것
- 마지막 스토리보드는 대표 이미지 생성에 바로 쓸 수 있을 정도로 구체적일 것

---------- TextMessage (scenario_writer_agent) ----------
서브스크립트 1  
Plot: 윤하는 네온 간판이 반짝이는 홍대 뒷골목에서 기억 배달 임무를 수행한다. 그녀는 작은 메모리 캐리어를 전달하던 중, 자신의 이름과 생년월일이 암호화된 라벨이 붙은 수상한 패키지를 발견한다. 패키지의 내용은 흐릿한 영상 조각으로, 어린 시절의 그녀와 모르는 여성이 함께 웃는 모습이 스쳐 지나간다. 윤하는 이 조각이 자신의 잃어버린 기억 일부와 연결되어 있음을 직감하고, 패키지를 열어 보려는 유혹을 뿌리치며 계속 임무를 이어가지만, 마음속에 불안이 

/usr/local/lib/python3.12/dist-packages/autogen_agentchat/agents/_assistant_agent.py:1109: UserWarning: Resolved model mismatch: nvidia/nemotron-3-super-120b-a12b:free != nvidia/nemotron-3-super-120b-a12b-20230311:free. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to nvidia/nemotron-3-super-120b-a12b-20230311:free to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- TextMessage (scene_planner_agent) ----------
장면 1  
Plot: 네온 간판이 반짝이는 홍대 뒷골목에서 기억 배달부 윤하는 routine 임무를 수행하던 중, 자신의 이름과 생년월일이 암호화된 라벨이 붙은 수상한 패키지를 발견한다. 패키지 안에는 흐릿한 영상 조각이 담겨 있어 어린 시절의 그녀와 모르는 여성이 함께 웃는 모습이 스쳐 지나간다. 윤하는 이 조각이 자신의 잃어버린 기억 일부와 연결됨을 직감하고, 호기심을 억누르며 임무를 이어가지만 가슴에 불안이 싹튼다.  
Involving Characters: 윤하  

장면 2  
Plot: 윤하는 패키지의 해킹 흔적을 추적하기 위해 불법 기억 복원 전문가 도윤을 찾아 narrow 반지하 작업실로 간다. 도윤은 빛나는 데이터 스트림 속에서 손상된 기억 조각을 재조합하며 냉소적이지만 정교한 손길을 보여준다. 윤하는 패키지의 암호를 해독해 달라고 부탁하고, 도윤은 기억 조각을 풀어낸다. 해독된 영상에서 윤하는母亲이 마지막에 남긴 말과 자신이 잊고 있던 사건의 단편을 보게 된다. 동시에, 서진이 이끄는 보안 드론이 그들의 위치를 감지하고 추격을 시작한다.  
Involving Characters: 윤하, 도윤, 서진  

장면 3  
Plot: 윤하와 도윤은 기억 패키지를 안전하게 옮기기 위해 한강변의 폐쇄된 전망대로 향한다. 거기서 그들은 mother와 관련된 마지막 기억을 되찾고 싶어 하는 어린 소녀 하나를 만난다. 하나는 윤하의 패키지 안에 그 기억이 담겨 있다고 믿는다. 윤하는 자신의 과거와 연결된 이 기억이 하나에게 줄 수 있는 위로라는 사실을 깨닫고, 서진의 추격을 피해 하나에게 기억을 전하려는 결심을 한다. 비가 내리는 밤, 네온빛이 물에 reflej되어 도시 전체가 반사판처럼 빛나는 가운데, 세 사람은 기억 전달을 위한 마지막 준비를 시작한다.  
Involving Characters: 윤하, 도윤, 하나, 서진 (추격 중)  

장면 4  
P

## 최종 스토리보드 확인

In [8]:
for message in result.messages:
    if message.source == "storyboard_artist_agent":
        print(message.content)

장면1  
Plot: 네온 간판이 반짝이는 홍대 뒷골목에서 기억 배달부 윤하는 routine 임무를 수행하던 중, 자신의 이름과 생년월일이 암호화된 라벨이 붙은 수상한 패키지를 발견한다. 패키지 안에는 흐릿한 영상 조각이 담겨 있어 어린 시절의 그녀와 모르는 여성이 함께 웃는 모습이 스쳐 지나간다. 윤하는 이 조각이 자신의 잃어버린 기억 일부와 연결됨을 직감하고, 호기심을 억누르며 임무를 이어가지만 가슴에 불안이 싹튼다.  
Scene Description: 비오는 밤, 홍대의 골목은 빨간 파란 네온 사인이 젖은 포장길에 반사돼 빛의 물결을 만든다. 윤하는 좁은 골목 사이를 빠르게 이동하며 손에 든 메모리 캐리어를 점검한다. 캐리어 옆에 놓여진 의문의 패키지는 검은 금속 케이스에 은은한 청빛 라벨이 붙어 있고, 라벨 속에 윤하의 이름과 생일이 암호화된 코드가 흐릿하게 빛난다. 패키지를 살짝 열어보니 내부에서 흐릿한 홀로그램 조각이 튀어나와, 어린 윤하와 미소 지은 desconocida 여성이 brevemente aparece y desaparece.  
Emotional Tone: 긴장감과 미묘한 향수, 알 수 없는 과거에 대한 불안이 뒤섞인 분위기.  
Visual Style: 사이버펑크 네온미학, 심도 shallow focus로 젖은 거리 reflections 강조, 색채는 차가운 파랑과 따뜻한 핑크 대비.  
Key Props: 메모리 캐리어, 암호 라벨이 붙은 검은 금속 패키지, 흐릿한 홀로그램 조각, 윤하의 우비와 손전등.  
Involving Characters: 윤하  
Cinematography Notes: 골목의 깊이를 보여주는 트래킹 샷으로 시작, 윤하의 얼굴에 클로즈업으로 불안 wyrażenie, 패키지 내부를 보여주는 매크로 샷으로 라벨과 홀로그램 디테일 강조, 배경 네온 빛이 bokeh 효과로 번짐.  

장면 2  
Plot: 윤하는 패키지의 해킹 흔적을 추적하기 위해 불법 기억 복원 전문가 도윤을 찾아 narrow 반지하 작업실로 간

In [9]:
def get_storyboard_text(result):
    storyboard_text = ""
    for message in result.messages:
        if message.source == "storyboard_artist_agent":
            storyboard_text = message.content

    return storyboard_text.replace("TERMINATE", "").strip()

storyboard_text = get_storyboard_text(result)
print(storyboard_text)

장면1  
Plot: 네온 간판이 반짝이는 홍대 뒷골목에서 기억 배달부 윤하는 routine 임무를 수행하던 중, 자신의 이름과 생년월일이 암호화된 라벨이 붙은 수상한 패키지를 발견한다. 패키지 안에는 흐릿한 영상 조각이 담겨 있어 어린 시절의 그녀와 모르는 여성이 함께 웃는 모습이 스쳐 지나간다. 윤하는 이 조각이 자신의 잃어버린 기억 일부와 연결됨을 직감하고, 호기심을 억누르며 임무를 이어가지만 가슴에 불안이 싹튼다.  
Scene Description: 비오는 밤, 홍대의 골목은 빨간 파란 네온 사인이 젖은 포장길에 반사돼 빛의 물결을 만든다. 윤하는 좁은 골목 사이를 빠르게 이동하며 손에 든 메모리 캐리어를 점검한다. 캐리어 옆에 놓여진 의문의 패키지는 검은 금속 케이스에 은은한 청빛 라벨이 붙어 있고, 라벨 속에 윤하의 이름과 생일이 암호화된 코드가 흐릿하게 빛난다. 패키지를 살짝 열어보니 내부에서 흐릿한 홀로그램 조각이 튀어나와, 어린 윤하와 미소 지은 desconocida 여성이 brevemente aparece y desaparece.  
Emotional Tone: 긴장감과 미묘한 향수, 알 수 없는 과거에 대한 불안이 뒤섞인 분위기.  
Visual Style: 사이버펑크 네온미학, 심도 shallow focus로 젖은 거리 reflections 강조, 색채는 차가운 파랑과 따뜻한 핑크 대비.  
Key Props: 메모리 캐리어, 암호 라벨이 붙은 검은 금속 패키지, 흐릿한 홀로그램 조각, 윤하의 우비와 손전등.  
Involving Characters: 윤하  
Cinematography Notes: 골목의 깊이를 보여주는 트래킹 샷으로 시작, 윤하의 얼굴에 클로즈업으로 불안 wyrażenie, 패키지 내부를 보여주는 매크로 샷으로 라벨과 홀로그램 디테일 강조, 배경 네온 빛이 bokeh 효과로 번짐.  

장면 2  
Plot: 윤하는 패키지의 해킹 흔적을 추적하기 위해 불법 기억 복원 전문가 도윤을 찾아 narrow 반지하 작업실로 간

## 스토리보드 저장

현재 Nemotron 텍스트 모델 기준으로 구성되어 있으므로,
마지막 결과를 텍스트 파일로 저장해서 후속 작업에 활용할 수 있게 정리

In [10]:
with open("nemotron_storyboard.txt", "w", encoding="utf-8") as f:
    f.write(storyboard_text)

print("nemotron_storyboard.txt 저장 완료")

nemotron_storyboard.txt 저장 완료


In [11]:
storyboard_text[:2000]

'장면1  \nPlot: 네온 간판이 반짝이는 홍대 뒷골목에서 기억 배달부 윤하는 routine 임무를 수행하던 중, 자신의 이름과 생년월일이 암호화된 라벨이 붙은 수상한 패키지를 발견한다. 패키지 안에는 흐릿한 영상 조각이 담겨 있어 어린 시절의 그녀와 모르는 여성이 함께 웃는 모습이 스쳐 지나간다. 윤하는 이 조각이 자신의 잃어버린 기억 일부와 연결됨을 직감하고, 호기심을 억누르며 임무를 이어가지만 가슴에 불안이 싹튼다.  \nScene Description: 비오는 밤, 홍대의 골목은 빨간 파란 네온 사인이 젖은 포장길에 반사돼 빛의 물결을 만든다. 윤하는 좁은 골목 사이를 빠르게 이동하며 손에 든 메모리 캐리어를 점검한다. 캐리어 옆에 놓여진 의문의 패키지는 검은 금속 케이스에 은은한 청빛 라벨이 붙어 있고, 라벨 속에 윤하의 이름과 생일이 암호화된 코드가 흐릿하게 빛난다. 패키지를 살짝 열어보니 내부에서 흐릿한 홀로그램 조각이 튀어나와, 어린 윤하와 미소 지은 desconocida 여성이 brevemente aparece y desaparece.  \nEmotional Tone: 긴장감과 미묘한 향수, 알 수 없는 과거에 대한 불안이 뒤섞인 분위기.  \nVisual Style: 사이버펑크 네온미학, 심도 shallow focus로 젖은 거리 reflections 강조, 색채는 차가운 파랑과 따뜻한 핑크 대비.  \nKey Props: 메모리 캐리어, 암호 라벨이 붙은 검은 금속 패키지, 흐릿한 홀로그램 조각, 윤하의 우비와 손전등.  \nInvolving Characters: 윤하  \nCinematography Notes: 골목의 깊이를 보여주는 트래킹 샷으로 시작, 윤하의 얼굴에 클로즈업으로 불안 wyrażenie, 패키지 내부를 보여주는 매크로 샷으로 라벨과 홀로그램 디테일 강조, 배경 네온 빛이 bokeh 효과로 번짐.  \n\n장면 2  \nPlot: 윤하는 패키지의 해킹 흔적을 추적하기 위해 불법 기억 복원 전문가 도윤을 찾아 narrow

보내주신 API key는 크레딧 모두 소진했다는 오류가 떠서 Nemotron3 라는 nvidia의 모델 무료 버전 api를 사용했습니다.

아무래도 무료 버전이다보니 이미지 생성이 어렵고 모델 자체 성능도 떨어져서 스크립트도 퀄리티가 떨어지는 것 같습니다.